# W&B Reports API — Programmatically Curate a Report

Reports let you pin curated runs, charts, and prose into a single
shareable page — the artifact that turns a scratchpad of experiments
into stakeholder-facing communication. Building them **programmatically**
means the same code can regenerate them after every training pass, so
your dashboards stay in sync with your latest runs by construction.

This notebook builds the shareable narrative view for the workshop pipeline.
It always includes the `tiny-transolver-train` run from
`experiment_tracking.ipynb`; if `sweeps.ipynb` has also run, the same
project will contain the fine-tune trials and promotion run linked in
the W&B lineage graph.

**Prerequisites**
- `uv sync` (adds `wandb-workspaces` to your environment).
- `uv run wandb login`.
- `experiment_tracking.ipynb` has been executed end-to-end at least
  once, so `wandb_emea/jb_physics_demo` contains a training run.
- Optional but recommended: run `sweeps.ipynb` first so the report
  sits downstream of the tuned-checkpoint promotion in the workshop
  story.


## 1. Setup

Imports and the target project. `wandb-workspaces` is a separate
package that wraps the Reports REST API; the workspaces module is
imported as `wr` by convention.


In [ ]:
import os
from pathlib import Path

import wandb
import wandb_workspaces.reports.v2 as wr



def load_local_env(filename: str = ".env") -> None:
    """Load simple KEY=VALUE lines from the nearest .env without echoing secrets."""
    for directory in [Path.cwd(), *Path.cwd().parents]:
        env_path = directory / filename
        if not env_path.exists():
            continue
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip("'\""))
        return


load_local_env()

ENTITY = os.getenv("WANDB_ENTITY", "wandb_emea")
PROJECT = os.getenv("WANDB_PROJECT", "jb_physics_demo")
if "/" in PROJECT:
    PROJECT = PROJECT.split("/")[-1]

print(f"wandb            = {wandb.__version__}")
print(f"target project   = {ENTITY}/{PROJECT}")


## 2. Build the report

A `Report` is a title, a description, and an ordered list of `blocks`.
Two kinds of block matter here:

- **Text blocks** (`H1`, `H2`, `P`, `MarkdownBlock`) — static prose
  authored inline, exactly what you would write in a markdown cell.
- **`PanelGrid`** — the live block. It holds one or more `Runset`s
  (each a query into a specific entity/project, optionally filtered by
  run-name substring) and one or more `Panel`s (`LinePlot`, `BarPlot`,
  `ScatterPlot`, `MediaBrowser`, ...). Panels re-query at render time,
  so runs logged **after** you save the report show up automatically.


In [ ]:
panel_grid = wr.PanelGrid(
    runsets=[
        wr.Runset(
            entity=ENTITY,
            project=PROJECT,
            name="TinyTransolver training",
            query="tiny-transolver",           # substring match on run name
        ),
    ],
    panels=[
        wr.LinePlot(
            title="Validation loss decomposition",
            x="epoch",
            y=["val/loss", "val/mse", "val/aux"],
            smoothing_factor=0.3,
        ),
        wr.LinePlot(
            title="Field accuracy",
            x="epoch",
            y=["val/pressure_rmse", "val/velocity_rmse", "val/field_relative_l2"],
        ),
        wr.LinePlot(
            title="Physics-weighted objective",
            x="epoch",
            y=["train/field_mse", "train/weighted_field_mse", "val/weighted_mse"],
        ),
        wr.LinePlot(
            title="TinyTransolver vs pointwise baseline",
            x="epoch",
            y=["val/pressure_rmse", "baseline/val_pressure_rmse", "model/improvement_vs_pointwise_pressure_rmse"],
        ),
        wr.LinePlot(
            title="Composite model-selection score",
            x="epoch",
            y=["model/selection_score", "baseline/selection_score", "model/selection_margin_vs_baseline"],
        ),
        wr.LinePlot(
            title="Design proxy metrics",
            x="epoch",
            y=["val/lift_proxy_mae", "val/drag_proxy_mae", "val/lift_proxy_spearman"],
        ),
        wr.LinePlot(
            title="Slice-routing health",
            x="epoch",
            y=["slice/point_entropy", "slice/usage_entropy", "slice/effective_slices"],
        ),
        wr.MediaBrowser(
            media_keys=[
                "dataset/canonical_airfoil_profile",
                "workshop/airfoil_profile_reference",
                "workshop/airfoil_profile_gallery",
                "summary/scorecard_epoch_15",
                "val/comparison_epoch_15",
                "val/error_map_epoch_15",
                "model/slice_usage_epoch_15",
                "model/slice_usage_evolution_video",
                "workshop/slice_usage_video",
                "model/slice_assignment_epoch_15",
                "model/slice_assignment_evolution_video",
                "workshop/slice_assignment_video",
            ],
            num_columns=2,
        ),
    ],
)

blocks = [
    wr.H1(text="TinyTransolver — training summary"),
    wr.P(
        text=(
            "Auto-generated summary of runs produced by "
            "experiment_tracking.ipynb. Panels re-query live at render "
            "time, so new runs appear here without regenerating the report."
        )
    ),
    wr.H2(text="Field accuracy, baseline, and routing health"),
    panel_grid,
    wr.MarkdownBlock(
        text=(
            "**Reading guide.** `val/loss` combines the weighted field MSE and slice regularisation. "
            "The pointwise baseline has no cross-point communication, so "
            "`model/selection_margin_vs_baseline` is the quick "
            "guardrail check that Physics-Attention is buying something. "
            "The lift/drag values are labelled proxies because the workshop "
            "dataset is analytic point-cloud data, not a CFD mesh with true "
            "surface integrals. The media browser shows scorecards, pressure "
            "airfoil profiles, comparisons, error maps, and routing images logged as `wandb.Image`s, "
            "plus `wandb.Video` GIFs of slice usage and slice assignments over training; "
            "the run also contains rich `val/case_diagnostics_epoch_*` Tables and artifact-backed "
            "workshop Tables: `workshop_airfoil_profiles:latest`, "
            "`workshop_case_diagnostics:latest`, `workshop_slice_usage_frames:latest`, "
            "and `workshop_slice_assignment_frames:latest`."
        )
    ),
]


## 3. Save and share

`report.save()` publishes the report to the target project as a draft.
`report.url` returns the shareable link — the same URL you would get
from clicking **Create report** in the W&B UI.


In [ ]:
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="TinyTransolver — training summary",
    description="Programmatically generated by notebooks/reports.ipynb.",
    blocks=blocks,
)

report.save()

print(report.url)
